# 파일명 수정 + 중복 제거
1. 정면/측면 실제 각도 판별 → 파일명 올바르게 수정
2. 중복 사진 삭제
> **시작 전**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택

## Step 1. 패키지 설치

In [ ]:
!pip install -q gdown insightface onnxruntime-gpu imagehash
import torch
print('✅ 설치 완료')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ 없음"}')

## Step 2. 설정

In [ ]:
# Google Drive 공유링크
DRIVE_SHARE_URL = 'https://drive.google.com/file/d/여기에_링크/view?usp=sharing'

ZIP_PATH  = '/content/dataset.zip'
RAW_DIR   = '/content/raw'
OUT_DIR   = '/content/output'   # 결과 저장 위치

YAW_THR   = 45.0   # 정면 판정 yaw 각도 (±45° 이내 = 정면)

from pathlib import Path
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print('✅ 설정 완료')

## Step 3. Drive에서 zip 다운로드 + 압축 해제

In [ ]:
import gdown, zipfile
from pathlib import Path

file_id = DRIVE_SHARE_URL.split('/d/')[1].split('/')[0]
download_url = f'https://drive.google.com/uc?id={file_id}'

print('다운로드 중...')
gdown.download(download_url, ZIP_PATH, quiet=False)

print('압축 해제 중...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(RAW_DIR)

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
all_imgs = [p for p in Path(RAW_DIR).rglob('*') if p.suffix.lower() in IMG_EXTS]
print(f'✅ 총 {len(all_imgs)}장 확인')
print('샘플:')
for p in sorted(all_imgs)[:5]:
    print(f'  {p.name}')

## Step 4. 인물별 그룹핑 확인

In [ ]:
from collections import defaultdict

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
raw_root = Path(RAW_DIR)
person_images = defaultdict(list)

for p in raw_root.rglob('*'):
    if p.suffix.lower() not in IMG_EXTS: continue
    person = p.parent.name
    person_images[person].append(p)

print(f'감지된 인물: {len(person_images)}명\n')
for person, imgs in sorted(person_images.items()):
    print(f'  {person}: {len(imgs)}장')

## Step 5. 모델 로드

In [ ]:
from insightface.app import FaceAnalysis
from imagehash import phash
from PIL import Image as PILImage
import cv2, numpy as np, shutil

face_app = FaceAnalysis(name='buffalo_l',
                        providers=['CUDAExecutionProvider','CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(640,640))
print('✅ insightface 로드')
print('✅ imagehash 로드')

## Step 6. 파일명 수정 + 중복 제거 실행
- 실제 yaw 각도로 정면/측면 판별 → 파일명 수정
- pHash로 완전 중복 제거

In [ ]:
import unicodedata

total_renamed  = 0
total_dupes    = 0
total_no_face  = 0
total_saved    = 0

for person, img_paths in sorted(person_images.items()):
    print(f'\n[{person}] {len(img_paths)}장 처리 중...')

    out_person_dir = Path(OUT_DIR) / person
    out_person_dir.mkdir(parents=True, exist_ok=True)

    seen_hashes = {}  # pHash 중복 체크
    cnt = defaultdict(int)  # {angle: count}
    renamed = 0
    dupes   = 0
    no_face = 0

    for img_path in sorted(img_paths):
        img = cv2.imread(str(img_path))
        if img is None: no_face += 1; continue

        # pHash 중복 체크
        try:
            h = str(phash(PILImage.open(img_path)))
            if h in seen_hashes:
                dupes += 1
                continue  # 중복이면 스킵
            seen_hashes[h] = str(img_path)
        except:
            pass

        # 얼굴 감지 + yaw 각도로 정면/측면 판별
        faces = face_app.get(img)
        if len(faces) == 0:
            no_face += 1
            continue

        f = faces[0]
        yaw = float(f.pose[1]) if hasattr(f, 'pose') else 0.0
        actual_angle = '정면' if abs(yaw) <= YAW_THR else '측면'

        # 파일명에 적힌 각도 확인
        stem = img_path.stem
        if '정면' in stem: label_angle = '정면'
        elif '측면' in stem: label_angle = '측면'
        else: label_angle = '미표기'

        # 각도 다르면 카운트
        if label_angle != '미표기' and label_angle != actual_angle:
            renamed += 1

        # 새 파일명 부여: [이름]_[실제각도]_[인덱스].jpg
        cnt[actual_angle] += 1
        ext = img_path.suffix.lower() or '.jpg'
        new_name = f'{person}_{actual_angle}_{cnt[actual_angle]:03d}{ext}'
        new_name = unicodedata.normalize('NFC', new_name)

        shutil.copy2(str(img_path), out_person_dir / new_name)
        total_saved += 1

    total_renamed += renamed
    total_dupes   += dupes
    total_no_face += no_face
    saved = cnt.get('정면', 0) + cnt.get('측면', 0)
    print(f'  저장={saved}장  각도수정={renamed}장  중복제거={dupes}장  얼굴없음={no_face}장')

print(f'\n🎉 완료!')
print(f'  최종 저장:  {total_saved}장')
print(f'  각도 수정:  {total_renamed}장')
print(f'  중복 제거:  {total_dupes}장')
print(f'  얼굴 없음:  {total_no_face}장')

## Step 7. 다운로드

In [ ]:
import shutil
from google.colab import files

print('zip 압축 중...')
shutil.make_archive('/content/output', 'zip', OUT_DIR)
print('✅ 압축 완료')
files.download('/content/output.zip')